# Active Layer Dynamics Index (ALDI) Calculator


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import rankdata, norm
from scipy.interpolate import PchipInterpolator
import zlib
import warnings
warnings.filterwarnings('ignore')


def sign3(x):
    """3-way sign function: returns 1, -1, or 0."""
    if x > 0:
        return 1
    elif x < 0:
        return -1
    return 0


class ActiveLayerDynamicsIndex:

    def __init__(self, min_years=8, bootstrap_n=1000,
                 alpha=0.10,  # alpha=0.10 to reduce Type II error; classification is slope-based
                 arctic_only=True, fill_gaps=True, max_gap_fill=2,
                 random_seed=42,
                 low_power_span=12, low_power_n=10,
                 max_min_slope=5.0,
                 boot_over_resid_max=4.0):

        self.min_years = min_years
        self.bootstrap_n = bootstrap_n
        self.alpha = alpha
        self.arctic_only = arctic_only
        self.fill_gaps = fill_gaps
        self.max_gap_fill = max_gap_fill
        self.random_seed = random_seed

        # Low power thresholds
        self.low_power_span = low_power_span
        self.low_power_n = low_power_n

        # Min slope safeguards
        self.max_min_slope = max_min_slope          # beta_cap = 5.0 cm/yr
        self.boot_over_resid_max = boot_over_resid_max  # kappa = 4

        self.bootstrap_cache = {}

        # Thresholds to be calibrated from data
        self.RAPID_THRESHOLD = None
        self.GRADUAL_THRESHOLD = None
        self._thresholds_calibrated = False

    def _site_seed(self, site_id, offset=0):
        """Site-specific seed using stable CRC32 hash."""
        h = zlib.crc32(site_id.encode('utf-8')) & 0xffffffff
        return self.random_seed + (h % 1_000_000) + offset

    # =========================================================================
    # ROBUST SIGMA ESTIMATION
    # =========================================================================

    def _robust_sigma(self, x):
        """Robust scale estimate using MAD, falling back to IQR."""
        x = np.asarray(x)
        x = x[np.isfinite(x)]
        if len(x) < 2:
            return np.nan
        med = np.median(x)
        mad = np.median(np.abs(x - med))
        if mad > 0:
            return 1.4826 * mad
        q75, q25 = np.percentile(x, [75, 25])
        iqr = q75 - q25
        if iqr > 0:
            return 0.7413 * iqr
        return np.std(x, ddof=1)

    # =========================================================================
    # DATA LOADING
    # =========================================================================

    def load_and_clean_data(self, filepath):
        """Load and preprocess CALM data with quality control."""
        df = pd.read_csv(filepath, encoding='latin-1')

        n_before = len(df.reset_index(drop=True))

        # Handle censored values (>130 means "exceeded probe length")
        df['Max'] = df['Max'].replace('>130', '135')  # Conservative replacement
        df['Max'] = df['Max'].replace('NA', np.nan)
        df['Max'] = pd.to_numeric(df['Max'], errors='coerce')
        # df = df[df['Max'] < 1000] #do not remove higher than 1000cm

        # Site ID: fixed-precision formatting
        df['site_id'] = (
            df['Lat'].map(lambda v: np.format_float_positional(v, precision=8, unique=False, trim='k'))
            + '_' +
            df['Long'].map(lambda v: np.format_float_positional(v, precision=8, unique=False, trim='k'))
        )
        df_clean = df.dropna(subset=['Max']).dropna(how='all').reset_index(drop=True)
        n_before_na = len(df_clean)
        # Deduplication: collapse true duplicates (same site_id + Year)
        df_clean = df_clean.groupby(['site_id', 'Year'], as_index=False).agg({
            'Max': 'mean',
            'Lat': 'first',
            'Long': 'first'
        }).reset_index(drop=True)
        n_after = len(df_clean)

        if self.arctic_only:
            df_clean = self._filter_arctic_sites(df_clean)

        print(f"  Raw observations: {n_before}")
        print(f"  After dropping missing rows: {n_before_na}")
        print(f"  Deduplicated: {n_before} -> {n_after} (collapsed {n_before - n_after} duplicates)")
        print(f"\n  Arctic filter (>=60N): {df_clean['site_id'].nunique()}/{df['site_id'].nunique()} sites")
        print(f"  Sites: {df_clean['site_id'].nunique()}")
        print(f"  Years: {int(df_clean['Year'].min())} - {int(df_clean['Year'].max())}")

        return df_clean

    def _filter_arctic_sites(self, df):
        """Filter for Arctic sites (>=60N)."""
        if 'Lat' not in df.columns:
            return df
        return df[df['Lat'] >= 60.0].copy()

    # =========================================================================
    # GAP FILLING
    # =========================================================================

    def fill_time_series_gaps(self, years, values, max_gap=None):
        """Fill small gaps using PCHIP interpolation."""
        if max_gap is None:
            max_gap = self.max_gap_fill

        years = np.array(years)
        values = np.array(values)
        sort_idx = np.argsort(years)
        years = years[sort_idx]
        values = values[sort_idx]

        year_min, year_max = int(years.min()), int(years.max())
        years_complete = np.arange(year_min, year_max + 1)

        # Identify gaps
        gaps = []
        for i in range(len(years) - 1):
            gap_size = years[i + 1] - years[i] - 1
            if gap_size > 0:
                gaps.append({
                    'start_year': int(years[i]),
                    'end_year': int(years[i + 1]),
                    'size': int(gap_size),
                    'fillable': gap_size <= max_gap
                })

        if not any(g['fillable'] for g in gaps):
            return {
                'years_filled': years,
                'values_filled': values,
                'n_interpolated': 0,
                'method': 'none'
            }

        interpolator = PchipInterpolator(years, values)
        years_filled = list(years)
        values_filled = list(values)
        n_interpolated = 0

        for year in years_complete:
            if year not in years:
                in_fillable_gap = False
                for gap in gaps:
                    if gap['fillable'] and gap['start_year'] < year < gap['end_year']:
                        in_fillable_gap = True
                        break
                if in_fillable_gap:
                    interp_value = float(interpolator(year))
                    values_filled.append(interp_value)
                    years_filled.append(year)
                    n_interpolated += 1

        # Sort chronologically
        sort_idx = np.argsort(years_filled)
        return {
            'years_filled': np.array(years_filled)[sort_idx],
            'values_filled': np.array(values_filled)[sort_idx],
            'n_interpolated': n_interpolated,
            'method': 'pchip'
        }

    # =========================================================================
    # COMPLETENESS
    # =========================================================================

    def calculate_completeness(self, years):
        """Calculate temporal completeness.

        Returns:
            completeness: fraction of years with data (n_obs / span)
            max_gap_spacing: maximum year-to-year spacing
            n_observations: count of data points
            span_years: total span from first to last year
        """
        years = np.array(years)
        span = int(years.max() - years.min() + 1)
        n_obs = len(years)
        completeness = n_obs / span if span > 0 else 0

        years_sorted = np.sort(years)
        gaps = np.diff(years_sorted)
        max_gap_spacing = int(gaps.max()) if len(gaps) > 0 else 0

        return {
            'completeness': completeness,
            'max_gap_spacing': max_gap_spacing,
            'n_observations': n_obs,
            'span_years': span
        }

    # =========================================================================
    # SEN'S SLOPE & MANN-KENDALL
    # =========================================================================

    def _calculate_sen_slope(self, years, values):
        """Calculate Sen's slope estimator."""
        n = len(values)
        slopes = []
        for i in range(n):
            for j in range(i + 1, n):
                if years[j] != years[i]:
                    slope = (values[j] - values[i]) / (years[j] - years[i])
                    slopes.append(slope)
        return np.median(slopes) if slopes else 0.0

    def _mann_kendall_statistic(self, values):
        """Calculate Mann-Kendall S statistic."""
        n = len(values)
        S = 0
        for i in range(n):
            for j in range(i + 1, n):
                S += np.sign(values[j] - values[i])
        return S

    def _hamed_rao_variance(self, values, S):
        """Hamed-Rao variance correction for autocorrelation."""
        n = len(values)

        unique, counts = np.unique(values, return_counts=True)
        tie_correction = sum(t * (t - 1) * (2 * t + 5) for t in counts if t > 1)
        var_S_base = (n * (n - 1) * (2 * n + 5) - tie_correction) / 18

        if n < 10:
            return var_S_base

        ranks = rankdata(values)

        n_s_ratio = 0
        for lag in range(1, min(n // 3, 10)):
            r_lag = np.corrcoef(ranks[:-lag], ranks[lag:])[0, 1]
            if np.isfinite(r_lag):
                weight = (n - lag) * (n - lag - 1) * (n - lag - 2)
                n_s_ratio += weight * r_lag

        correction = 1 + (2 / (n * (n - 1) * (n - 2))) * n_s_ratio
        correction = max(0.1, min(correction, 10.0))

        return var_S_base * correction

    def mann_kendall_test(self, years, values):
        """Mann-Kendall test with Hamed-Rao variance correction."""
        n = len(values)
        if n < 4:
            return {'mk_pvalue': 1.0, 'mk_z': 0.0, 'sen_slope': 0.0,
                    'S_statistic': 0, 'var_S': 0.0}

        S = self._mann_kendall_statistic(values)
        var_S = self._hamed_rao_variance(values, S)

        if S > 0:
            Z = (S - 1) / np.sqrt(var_S)
        elif S < 0:
            Z = (S + 1) / np.sqrt(var_S)
        else:
            Z = 0

        p_value = 2 * (1 - norm.cdf(abs(Z)))
        sen_slope = self._calculate_sen_slope(years, values)

        return {
            'mk_pvalue': p_value,
            'mk_z': Z,
            'sen_slope': sen_slope,
            'S_statistic': S,
            'var_S': var_S
        }

    # =========================================================================
    # MIN MEANINGFUL SLOPE (beta_min)
    # =========================================================================

    def calculate_min_meaningful_slope_residual(self, years, values):
        """Residual-based noise floor estimation (beta_resid)."""
        years = np.array(years)
        values = np.array(values)
        n = len(years)
        span = float(years.max() - years.min() + 1)

        if n < 3:
            return np.nan

        sen_slope = self._calculate_sen_slope(years, values)
        detrended = values - sen_slope * (years - years[0])

        diffs = np.diff(detrended)
        if len(diffs) < 2:
            return np.nan

        med_d = np.median(diffs)
        mad_d = np.median(np.abs(diffs - med_d))
        sigma_d = 1.4826 * mad_d if mad_d > 0 else np.std(diffs, ddof=1)

        sigma_winsor = sigma_d
        if sigma_d > 0:
            low, high = med_d - 3 * sigma_d, med_d + 3 * sigma_d
            diffs_clip = np.clip(diffs, low, high)
            sigma_winsor = np.std(diffs_clip, ddof=1)

        sigma_pt = sigma_winsor / np.sqrt(2) if sigma_winsor > 0 else sigma_d / np.sqrt(2)

        min_slope = (2 * sigma_pt) / (span * np.sqrt(n))

        return min(min_slope, self.max_min_slope)

    def calculate_min_meaningful_slope_bootstrap(self, bootstrap_results):
        """Bootstrap-based noise floor using robust MAD (sigma_boot)."""
        sen_draws = bootstrap_results.get('sen_draws', np.array([]))
        x = sen_draws[np.isfinite(sen_draws)]
        if len(x) < 10:
            return np.nan

        med = np.median(x)
        s = self._robust_sigma(x)
        if np.isfinite(s) and s > 0:
            x = np.clip(x, med - 4 * s, med + 4 * s)

        sigma_slope = self._robust_sigma(x)
        return sigma_slope if np.isfinite(sigma_slope) else np.nan

    def calculate_min_meaningful_slope(self, years, values, bootstrap_results):
        """
        Combined beta_min with guarded maximum (Eq. 3 in methods).

        beta_min = min(max(beta_base, sigma_boot if <= kappa*beta_base else beta_base), beta_cap)
        where beta_base = max(beta_resid, beta_quant)
        """
        years = np.array(years)
        values = np.array(values)

        min_slope_resid = self.calculate_min_meaningful_slope_residual(years, values)
        if not np.isfinite(min_slope_resid):
            min_slope_resid = 0.0

        min_slope_boot = self.calculate_min_meaningful_slope_bootstrap(bootstrap_results)

        # Quantization floor
        diffs = np.abs(np.diff(np.sort(values)))
        diffs = diffs[diffs > 0]
        q = np.median(diffs) if len(diffs) > 0 else 1.0
        span_years = float(years.max() - years.min() + 1)
        quant_floor = 0.5 * q / max(span_years, 1.0)

        # Combine with safeguards
        min_base = max(quant_floor, min_slope_resid)

        if np.isfinite(min_slope_boot):
            if min_slope_boot <= self.boot_over_resid_max * min_base:
                min_slope = max(min_base, min_slope_boot)
            else:
                min_slope = min_base
        else:
            min_slope = min_base

        # Final hard cap
        min_slope = min(min_slope, self.max_min_slope)

        return min_slope, min_slope_resid, min_slope_boot, quant_floor

    # =========================================================================
    # TREND COMPONENT
    # =========================================================================

    def calculate_trend_component(self, site_id, years, values, bootstrap_results):
        """Calculate trend statistics: MK, Sen's slope, beta_min, CIs."""
        if len(years) < self.min_years:
            return self._empty_trend_component()

        years = np.array(years)
        values = np.array(values)
        n = len(years)
        span = int(years.max() - years.min() + 1)

        mk_result = self.mann_kendall_test(years, values)
        sen_slope = mk_result['sen_slope']
        mk_pvalue = mk_result['mk_pvalue']

        min_slope, min_slope_resid, min_slope_boot, quant_floor = \
            self.calculate_min_meaningful_slope(years, values, bootstrap_results)

        # Sen slope CI from bootstrap
        sen_draws = bootstrap_results.get('sen_draws', np.array([]))
        valid_draws = sen_draws[np.isfinite(sen_draws)]
        if len(valid_draws) >= 100:
            ci_lower_pct = 100 * (self.alpha / 2)
            ci_upper_pct = 100 * (1 - self.alpha / 2)
            sen_ci_lower = np.percentile(valid_draws, ci_lower_pct)
            sen_ci_upper = np.percentile(valid_draws, ci_upper_pct)
            sen_ci_excludes_zero = (sen_ci_lower > 0) or (sen_ci_upper < 0)
        else:
            sen_ci_lower = np.nan
            sen_ci_upper = np.nan
            sen_ci_excludes_zero = False

        # Significance
        trend_significant = mk_pvalue < self.alpha or sen_ci_excludes_zero
        slope_meaningful = abs(sen_slope) >= min_slope

        has_longterm_direction = trend_significant and slope_meaningful
        low_power = span < self.low_power_span or n < self.low_power_n

        return {
            'sen_slope': sen_slope,
            'mk_pvalue': mk_pvalue,
            'mk_z': mk_result['mk_z'],
            'sen_ci_lower': sen_ci_lower,
            'sen_ci_upper': sen_ci_upper,
            'sen_ci_excludes_zero': sen_ci_excludes_zero,
            'min_meaningful_slope': min_slope,
            'min_slope_residual': min_slope_resid,
            'min_slope_bootstrap': min_slope_boot if np.isfinite(min_slope_boot) else np.nan,
            'quant_floor': quant_floor,
            'trend_significant': trend_significant,
            'slope_meaningful': slope_meaningful,
            'has_longterm_direction': has_longterm_direction,
            'low_power': low_power,
            'n_years': n,
            'span_years': span
        }

    def _empty_trend_component(self):
        return {
            'sen_slope': np.nan, 'mk_pvalue': np.nan, 'mk_z': np.nan,
            'sen_ci_lower': np.nan, 'sen_ci_upper': np.nan,
            'sen_ci_excludes_zero': False,
            'min_meaningful_slope': np.nan, 'min_slope_residual': np.nan,
            'min_slope_bootstrap': np.nan, 'quant_floor': np.nan,
            'trend_significant': False, 'slope_meaningful': False,
            'has_longterm_direction': False, 'low_power': True,
            'n_years': 0, 'span_years': 0
        }

    # =========================================================================
    # REVERSAL DETECTION
    # =========================================================================

    def calculate_reversal_component(self, site_id, years, values, bootstrap_results,
                                     min_slope, early_end, recent_start):
        """
        Detect trend reversal between early and recent windows.

        A site is classified as transitional when all three criteria are met:
          (i)   sgn(beta_early) != sgn(beta_recent)
          (ii)  Both bootstrap CIs exclude zero
          (iii) Both |beta| > max(theta_gradual, beta_min)
        """
        years = np.array(years)
        values = np.array(values)

        # Guard against overlapping windows
        if recent_start <= early_end:
            return self._empty_reversal_component()

        # Use FIXED cutoffs (same as bootstrap)
        early_mask = years <= early_end
        early_years = years[early_mask]
        early_values = values[early_mask]

        recent_mask = years >= recent_start
        recent_years = years[recent_mask]
        recent_values = values[recent_mask]

        # Require at least 5 observations in each window
        if len(recent_years) < 5 or len(early_years) < 5:
            return self._empty_reversal_component()

        early_sen_slope = self._calculate_sen_slope(early_years, early_values)
        recent_sen_slope = self._calculate_sen_slope(recent_years, recent_values)

        # Bootstrap CI for early slope
        early_draws = bootstrap_results.get('early_sen_draws', np.array([]))
        valid_early = early_draws[np.isfinite(early_draws)]
        if len(valid_early) >= 100:
            ci_lower_pct = 100 * (self.alpha / 2)
            ci_upper_pct = 100 * (1 - self.alpha / 2)
            early_ci_lower = np.percentile(valid_early, ci_lower_pct)
            early_ci_upper = np.percentile(valid_early, ci_upper_pct)
            early_ci_excludes_zero = (early_ci_lower > 0) or (early_ci_upper < 0)
        else:
            early_ci_lower = np.nan
            early_ci_upper = np.nan
            early_ci_excludes_zero = False

        # Bootstrap CI for recent slope
        recent_draws = bootstrap_results.get('recent_sen_draws', np.array([]))
        valid_recent = recent_draws[np.isfinite(recent_draws)]
        if len(valid_recent) >= 100:
            ci_lower_pct = 100 * (self.alpha / 2)
            ci_upper_pct = 100 * (1 - self.alpha / 2)
            recent_ci_lower = np.percentile(valid_recent, ci_lower_pct)
            recent_ci_upper = np.percentile(valid_recent, ci_upper_pct)
            recent_ci_excludes_zero = (recent_ci_lower > 0) or (recent_ci_upper < 0)
        else:
            recent_ci_lower = np.nan
            recent_ci_upper = np.nan
            recent_ci_excludes_zero = False

        # +1 = recent thickening, -1 = recent thinning, 0 = unclear
        if recent_ci_excludes_zero:
            recent_direction = sign3(recent_sen_slope)
        else:
            recent_direction = 0

        # Three reversal criteria
        # Criterion (iii): max(theta_gradual, beta_min) as in Table 1
        stationary_band = max(self.GRADUAL_THRESHOLD, min_slope)

        opposite_signs = (sign3(early_sen_slope) != sign3(recent_sen_slope)
                          and sign3(early_sen_slope) != 0
                          and sign3(recent_sen_slope) != 0)
        both_ci_exclude_zero = early_ci_excludes_zero and recent_ci_excludes_zero
        both_exceed_band = (abs(early_sen_slope) > stationary_band
                            and abs(recent_sen_slope) > stationary_band)

        reversal = opposite_signs and both_ci_exclude_zero and both_exceed_band

        return {
            'early_sen_slope': early_sen_slope,
            'early_ci_lower': early_ci_lower,
            'early_ci_upper': early_ci_upper,
            'early_ci_excludes_zero': early_ci_excludes_zero,
            'recent_sen_slope': recent_sen_slope,
            'recent_ci_lower': recent_ci_lower,
            'recent_ci_upper': recent_ci_upper,
            'recent_ci_excludes_zero': recent_ci_excludes_zero,
            'recent_direction': recent_direction,
            'reversal': reversal,
            'early_n': len(early_years),
            'recent_n': len(recent_years)
        }

    def _empty_reversal_component(self):
        return {
            'early_sen_slope': np.nan, 'early_ci_lower': np.nan,
            'early_ci_upper': np.nan, 'early_ci_excludes_zero': False,
            'recent_sen_slope': np.nan, 'recent_ci_lower': np.nan,
            'recent_ci_upper': np.nan, 'recent_ci_excludes_zero': False,
            'recent_direction': 0,
            'reversal': False,
            'early_n': 0, 'recent_n': 0
        }

    # =========================================================================
    # BOOTSTRAP
    # =========================================================================

    def bootstrap_site(self, site_id, years, values, early_end, recent_start):
        """
        Block bootstrap for Sen's slope and windowed slopes.

        Args:
            early_end: Fixed cutoff year for early window (years <= early_end)
            recent_start: Fixed cutoff year for recent window (years >= recent_start)

        Note: early_end and recent_start are defined from the ORIGINAL series
        and used consistently across all bootstrap replicates to avoid
        window-redefinition noise in CIs.
        """
        cache_key = site_id
        if cache_key in self.bootstrap_cache:
            return self.bootstrap_cache[cache_key]

        n = len(years)
        rng = np.random.default_rng(self._site_seed(site_id))

        sen_draws = []
        early_sen_draws = []
        recent_sen_draws = []

        # Adaptive block size: b = max(2, min(5, n/4))
        block_size = max(2, min(5, n // 4))
        n_blocks = int(np.ceil(n / block_size))

        for _ in range(self.bootstrap_n):
            block_starts = rng.integers(0, n - block_size + 1, size=n_blocks)
            indices = []
            for start in block_starts:
                indices.extend(range(start, min(start + block_size, n)))
            indices = np.array(indices[:n])

            boot_years = years[indices]
            boot_values = values[indices]

            sort_idx = np.argsort(boot_years)
            boot_years = boot_years[sort_idx]
            boot_values = boot_values[sort_idx]

            # Full-record Sen slope
            sen = self._calculate_sen_slope(boot_years, boot_values)
            sen_draws.append(sen)

            # Early Sen slope using FIXED cutoff from original series
            early_mask = boot_years <= early_end
            early_years_boot = boot_years[early_mask]
            early_values_boot = boot_values[early_mask]

            if len(early_years_boot) >= 5:
                early_sen = self._calculate_sen_slope(early_years_boot, early_values_boot)
            else:
                early_sen = np.nan
            early_sen_draws.append(early_sen)

            # Recent Sen slope using FIXED cutoff from original series
            recent_mask = boot_years >= recent_start
            recent_years_boot = boot_years[recent_mask]
            recent_values_boot = boot_values[recent_mask]

            if len(recent_years_boot) >= 5:
                recent_sen = self._calculate_sen_slope(recent_years_boot, recent_values_boot)
            else:
                recent_sen = np.nan
            recent_sen_draws.append(recent_sen)

        result = {
            'sen_draws': np.array(sen_draws),
            'early_sen_draws': np.array(early_sen_draws),
            'recent_sen_draws': np.array(recent_sen_draws),
        }

        self.bootstrap_cache[cache_key] = result
        return result

    # =========================================================================
    # THRESHOLD CALIBRATION
    # =========================================================================

    def calibrate_thresholds(self, df):
        """
        Calibrate GRADUAL and RAPID thresholds from data distribution.

        theta_gradual = clip(P50(|beta_Sen|), 0.10, 0.30) cm/yr
        theta_rapid   = clip(P80(|beta_Sen|), 0.50, 2.00) cm/yr
        theta_rapid  >= 2 * theta_gradual  (minimum separation)

        Bounds justification:
          - Lower bound (0.1-0.3 cm/yr): CALM measurement precision (~1-2 cm
            for probing/thaw-tube methods) accumulated over typical record lengths
          - Upper bound (0.5-2.0 cm/yr): prevents outliers from distorting
            thresholds

        Reference: Brown et al. (2000), Streletskiy et al. (2017)
        """
        print("\n  Calibrating thresholds from data...")

        all_slopes = []
        site_ids = df['site_id'].unique()

        for site_id in site_ids:
            site_data = df[df['site_id'] == site_id].sort_values('Year')
            if len(site_data) < self.min_years:
                continue

            years = site_data['Year'].to_numpy()
            values = site_data['Max'].to_numpy()

            # Use same gap-filling as classification for consistency
            if self.fill_gaps:
                gap = self.fill_time_series_gaps(years, values)
                years_use = gap['years_filled']
                values_use = gap['values_filled']
            else:
                years_use, values_use = years, values

            mk_result = self.mann_kendall_test(years_use, values_use)
            all_slopes.append(abs(mk_result['sen_slope']))

        all_slopes = np.array(all_slopes)

        # Percentile-based thresholds with literature-bounded clips
        gradual_raw = np.percentile(all_slopes, 50)
        rapid_raw = np.percentile(all_slopes, 80)

        self.GRADUAL_THRESHOLD = np.clip(gradual_raw, 0.10, 0.30)
        self.RAPID_THRESHOLD = np.clip(rapid_raw, 0.50, 2.00)

        # Enforce minimum separation: rapid >= 2 * gradual
        if self.RAPID_THRESHOLD < 2 * self.GRADUAL_THRESHOLD:
            self.RAPID_THRESHOLD = 2 * self.GRADUAL_THRESHOLD

        self._thresholds_calibrated = True

        print(f"  theta_gradual = {self.GRADUAL_THRESHOLD:.3f} cm/yr "
              f"(P50={gradual_raw:.3f}, bounded [0.10, 0.30])")
        print(f"  theta_rapid   = {self.RAPID_THRESHOLD:.3f} cm/yr "
              f"(P80={rapid_raw:.3f}, bounded [0.50, 2.00])")
        print(f"  Separation: rapid/gradual = {self.RAPID_THRESHOLD / self.GRADUAL_THRESHOLD:.1f}x")

    # =========================================================================
    # CLASSIFICATION
    # =========================================================================

    def classify_site(self, site_id, trend, reversal_result, bootstrap_results, min_slope):
        """
        Two-step classification + confidence + P(c).

        Step 1: Check reversal criteria -> transitional
        Step 2: Slope-based assignment using calibrated thresholds

        IMPORTANT: Significance is a CONFIDENCE FLAG, not a classification gate.
        This aligns with how ALT trends are reported in the literature
        (e.g., Biskaborn et al., 2019).

        Categories (based on slope magnitude):
          - rapid_thickening:   Sen > RAPID_THRESHOLD
          - gradual_thickening: Sen in GRADUAL-RAPID range
          - no_trend:           |slope| < GRADUAL_THRESHOLD
          - gradual_thinning:   Sen in -RAPID to -GRADUAL range
          - rapid_thinning:     Sen < -RAPID_THRESHOLD
          - transitional:       Detected reversal (trend direction change)

        Confidence levels:
          - high:     Significant AND exceeds noise floor
          - moderate: Significant OR exceeds noise floor
          - low:      Neither
        """
        if trend.get('n_years', 0) < self.min_years:
            return self._empty_classification()

        # Ensure thresholds are calibrated
        if not self._thresholds_calibrated:
            self.GRADUAL_THRESHOLD = 0.30
            self.RAPID_THRESHOLD = 1.00

        sen_slope = trend['sen_slope']
        trend_significant = trend['trend_significant']
        slope_meaningful = trend['slope_meaningful']
        low_power = trend['low_power']
        reversal = reversal_result['reversal']

        # Step 1: Reversal check
        if reversal:
            classification = 'transitional'
        # Step 2: Slope-based classification
        elif sen_slope > self.RAPID_THRESHOLD:
            classification = 'rapid_thickening'
        elif sen_slope > self.GRADUAL_THRESHOLD:
            classification = 'gradual_thickening'
        elif sen_slope < -self.RAPID_THRESHOLD:
            classification = 'rapid_thinning'
        elif sen_slope < -self.GRADUAL_THRESHOLD:
            classification = 'gradual_thinning'
        else:
            classification = 'no_trend'

        # Confidence level
        if trend_significant and slope_meaningful:
            confidence = 'high'
        elif trend_significant or slope_meaningful:
            confidence = 'moderate'
        else:
            confidence = 'low'

        # Bootstrap classification probabilities P(c)
        # Computed across 5 slope-based categories only;
        # transitional is deterministic and not assigned probabilistic weights
        sen_draws = bootstrap_results.get('sen_draws', np.array([]))
        valid_draws = sen_draws[np.isfinite(sen_draws)]

        if len(valid_draws) >= 100:
            p_rapid_thick = float(np.mean(valid_draws > self.RAPID_THRESHOLD))
            p_gradual_thick = float(np.mean(
                (valid_draws > self.GRADUAL_THRESHOLD) & (valid_draws <= self.RAPID_THRESHOLD)
            ))
            p_no_trend = float(np.mean(np.abs(valid_draws) <= self.GRADUAL_THRESHOLD))
            p_gradual_thin = float(np.mean(
                (valid_draws < -self.GRADUAL_THRESHOLD) & (valid_draws >= -self.RAPID_THRESHOLD)
            ))
            p_rapid_thin = float(np.mean(valid_draws < -self.RAPID_THRESHOLD))
            p_thickening = p_rapid_thick + p_gradual_thick
            p_thinning = p_gradual_thin + p_rapid_thin
        else:
            p_rapid_thick = p_gradual_thick = p_no_trend = np.nan
            p_gradual_thin = p_rapid_thin = np.nan
            p_thickening = p_thinning = np.nan

        # Evidence string
        evidence = []
        evidence.append(f'Sen={sen_slope:.3f} cm/yr')
        if trend_significant:
            evidence.append('SIG')
        if reversal:
            evidence.append('REVERSAL')
        if low_power:
            evidence.append('LOW_PWR')
        evidence_str = '; '.join(evidence)

        return {
            'classification': classification,
            'confidence': confidence,
            'p_rapid_thickening': p_rapid_thick,
            'p_gradual_thickening': p_gradual_thick,
            'p_thickening': p_thickening,
            'p_no_trend': p_no_trend,
            'p_gradual_thinning': p_gradual_thin,
            'p_rapid_thinning': p_rapid_thin,
            'p_thinning': p_thinning,
            # Flags
            'flag_significant': trend_significant,
            'flag_low_power': low_power,
            'evidence': evidence_str
        }

    def _empty_classification(self):
        return {
            'classification': 'insufficient_data',
            'confidence': 'none',
            'p_rapid_thickening': np.nan,
            'p_gradual_thickening': np.nan,
            'p_thickening': np.nan,
            'p_no_trend': np.nan,
            'p_gradual_thinning': np.nan,
            'p_rapid_thinning': np.nan,
            'p_thinning': np.nan,
            'flag_significant': False,
            'flag_low_power': True,
            'evidence': 'Insufficient data'
        }

    # =========================================================================
    # MAIN CALCULATION
    # =========================================================================

    def calculate_aldi_for_site(self, site_data):
        """Calculate complete ALDI for one site."""
        site_data = site_data.sort_values('Year')
        site_id = site_data['site_id'].iloc[0]
        years = site_data['Year'].values
        values = site_data['Max'].values

        # Gap filling
        if self.fill_gaps:
            gap_result = self.fill_time_series_gaps(years, values)
            years_analysis = gap_result['years_filled']
            values_analysis = gap_result['values_filled']
            n_interpolated = gap_result['n_interpolated']
            gap_fill_method = gap_result['method']
        else:
            years_analysis = years
            values_analysis = values
            n_interpolated = 0
            gap_fill_method = 'none'

        quality = self.calculate_completeness(years)

        # Define fixed time-based cutoffs for early/recent windows
        y0 = years_analysis.min()
        y1 = years_analysis.max()
        span = y1 - y0 + 1

        # For records >= 14 years: first and last 7 calendar years
        # For shorter records: proportionally scaled, minimum 5 years, no overlap
        if span >= 14:
            early_end = y0 + 6
            recent_start = y1 - 6
        else:
            half = max(3, int(span / 2) - 1)
            half = min(half, int((span - 1) // 2))
            early_end = y0 + half
            recent_start = y1 - half

        # Bootstrap with fixed cutoffs
        bootstrap_results = self.bootstrap_site(
            site_id, years_analysis, values_analysis,
            early_end=early_end, recent_start=recent_start
        )

        # Trend estimation
        trend = self.calculate_trend_component(site_id, years_analysis, values_analysis, bootstrap_results)

        min_slope = trend['min_meaningful_slope']

        # Reversal detection
        reversal_result = self.calculate_reversal_component(
            site_id, years_analysis, values_analysis, bootstrap_results, min_slope,
            early_end=early_end, recent_start=recent_start
        )

        # Classification
        classification = self.classify_site(
            site_id, trend, reversal_result, bootstrap_results, min_slope
        )

        # Assemble result
        result = {
            'site_id': site_id,
            'lat': site_data['Lat'].iloc[0],
            'lon': site_data['Long'].iloc[0],
            'n_observations': quality['n_observations'],
            'n_after_gapfill': len(years_analysis),
            'n_interpolated': n_interpolated,
            'gap_fill_method': gap_fill_method,
            'span_years': quality['span_years'],
            'completeness': quality['completeness'],
            'max_gap_spacing': quality['max_gap_spacing'],
            'year_min': int(years.min()),
            'year_max': int(years.max()),
            'mean_alt': np.mean(values_analysis),
            'median_alt': np.median(values_analysis),
            'std_alt': np.std(values_analysis, ddof=1),
            **{f'class_{k}': v for k, v in classification.items()},
            **{f'trend_{k}': v for k, v in trend.items()},
            **{f'rev_{k}': v for k, v in reversal_result.items()},
        }

        return result

    def calculate_all_sites(self, df):
        """Calculate ALDI for all qualifying sites."""
        # Calibrate thresholds first
        self.calibrate_thresholds(df)

        results = []
        site_ids = df['site_id'].unique()
        n_sites = len(site_ids)

        for i, site_id in enumerate(site_ids):
            site_data = df[df['site_id'] == site_id]

            if len(site_data) < self.min_years:
                continue

            if (i + 1) % 25 == 0:
                print(f"  {i + 1}/{n_sites} sites...")

            try:
                result = self.calculate_aldi_for_site(site_data)
                results.append(result)
            except Exception as e:
                print(f"  Error at {site_id}: {e}")
                continue

        return pd.DataFrame(results)


# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    print("=" * 70)
    print("ACTIVE LAYER DYNAMICS INDEX (ALDI) v4.0")
    print("=" * 70)
    print("\nMethods-aligned version (Section 2.1)")
    print("\nPRIMARY CATEGORIES (slope-based):")
    print("  rapid_thickening   | gradual_thickening | no_trend")
    print("  gradual_thinning   | rapid_thinning     | transitional")
    print("\nCONFIDENCE: high | moderate | low")
    print("FLAGS: significant, low_power")

    calc = ActiveLayerDynamicsIndex(
        min_years=8,
        bootstrap_n=1000,
        alpha=0.10,         # Reduces Type II error; classification is slope-based
        arctic_only=True,
        fill_gaps=True,
        max_gap_fill=2,
        random_seed=42,
        low_power_span=12,
        low_power_n=10,
        max_min_slope=5.0,
        boot_over_resid_max=4.0,
    )

    filepath = '/content/drive/MyDrive/UND/Index/CALM_ALT_all_Max.csv'
    print(f"\nLoading data from: {filepath}")
    df = calc.load_and_clean_data(filepath)

    print("\n" + "=" * 70)
    print("PROCESSING")
    print("=" * 70)
    print(f"\nProcessing {df['site_id'].nunique()} sites...")

    results = calc.calculate_all_sites(df)

    print(f"\n  Completed: {len(results)} sites")

    # Summary
    print("\n" + "=" * 70)
    print("CLASSIFICATION SUMMARY")
    print("=" * 70)
    class_counts = results['class_classification'].value_counts()
    for cls, count in class_counts.items():
        pct = 100 * count / len(results)
        print(f"  {cls:25s}: {count:3d} ({pct:5.1f}%)")

    # Aggregate thickening vs thinning
    thick_cats = ['rapid_thickening', 'gradual_thickening']
    thin_cats = ['rapid_thinning', 'gradual_thinning']
    n_thick = results['class_classification'].isin(thick_cats).sum()
    n_thin = results['class_classification'].isin(thin_cats).sum()
    n_no_trend = (results['class_classification'] == 'no_trend').sum()
    n_trans = (results['class_classification'] == 'transitional').sum()
    print(f"\n  THICKENING TOTAL:    {n_thick:3d} ({100 * n_thick / len(results):5.1f}%)")
    print(f"  THINNING TOTAL:      {n_thin:3d} ({100 * n_thin / len(results):5.1f}%)")
    print(f"  NO_TREND:            {n_no_trend:3d} ({100 * n_no_trend / len(results):5.1f}%)")
    print(f"  TRANSITIONAL:        {n_trans:3d} ({100 * n_trans / len(results):5.1f}%)")

    # Confidence levels
    print("\n" + "=" * 70)
    print("CONFIDENCE LEVELS")
    print("=" * 70)
    conf_counts = results['class_confidence'].value_counts()
    for conf, count in conf_counts.items():
        pct = 100 * count / len(results)
        print(f"  {conf:15s}: {count:3d} ({pct:5.1f}%)")

    # Flags
    print("\n" + "=" * 70)
    print("FLAGS")
    print("=" * 70)
    print(f"  significant: {results['class_flag_significant'].sum():3d} "
          f"({100 * results['class_flag_significant'].mean():.1f}%)")
    print(f"  low_power:   {results['class_flag_low_power'].sum():3d} "
          f"({100 * results['class_flag_low_power'].mean():.1f}%)")

    # Direction consistency check
    print("\n" + "=" * 70)
    print("DIRECTION CONSISTENCY")
    print("=" * 70)
    for cls in ['rapid_thickening', 'gradual_thickening', 'gradual_thinning',
                'rapid_thinning', 'no_trend', 'transitional']:
        subset = results[results['class_classification'] == cls]
        if len(subset) > 0:
            sen_pos = (subset['trend_sen_slope'] > 0).sum()
            sen_neg = (subset['trend_sen_slope'] < 0).sum()
            sen_zero = (np.abs(subset['trend_sen_slope']) < 0.001).sum()
            print(f"  {cls:25s}: Sen>0={sen_pos:2d}, Sen<0={sen_neg:2d}, Sen~0={sen_zero:2d}")

    # Calibrated thresholds
    print("\n" + "=" * 70)
    print("CALIBRATED THRESHOLDS")
    print("=" * 70)
    print(f"  GRADUAL_THRESHOLD: {calc.GRADUAL_THRESHOLD:.3f} cm/yr")
    print(f"  RAPID_THRESHOLD:   {calc.RAPID_THRESHOLD:.3f} cm/yr")
    print(f"  alpha:             {calc.alpha}")

    # Save
    output_path = '/content/drive/MyDrive/UND/Index/aldi_v4_results.csv'
    results.to_csv(output_path, index=False)
    print(f"\nResults saved to: {output_path}")
    print(f"Output columns: {len(results.columns)}")

    print("\n" + "=" * 70)
    print("ALDI v4.0 COMPLETE")
    print("=" * 70)

ACTIVE LAYER DYNAMICS INDEX (ALDI) v4.0

Methods-aligned version (Section 2.1)

PRIMARY CATEGORIES (slope-based):
  rapid_thickening   | gradual_thickening | no_trend
  gradual_thinning   | rapid_thinning     | transitional

CONFIDENCE: high | moderate | low
FLAGS: significant, low_power

Loading data from: /content/drive/MyDrive/UND/Index/CALM_ALT_all_Max.csv
  Raw observations: 3241
  After dropping missing rows: 3101
  Deduplicated: 3241 -> 3099 (collapsed 142 duplicates)

  Arctic filter (>=60N): 171/203 sites
  Sites: 171
  Years: 1962 - 2024

PROCESSING

Processing 171 sites...

  Calibrating thresholds from data...
  theta_gradual = 0.300 cm/yr (P50=0.444, bounded [0.10, 0.30])
  theta_rapid   = 1.310 cm/yr (P80=1.310, bounded [0.50, 2.00])
  Separation: rapid/gradual = 4.4x
  25/171 sites...
  50/171 sites...
  75/171 sites...
  100/171 sites...
  150/171 sites...

  Completed: 129 sites

CLASSIFICATION SUMMARY
  no_trend                 :  51 ( 39.5%)
  gradual_thickening     